# TWAS, Colocalization & ColocBoost

This notebook demonstrates four downstream analyses that integrate molecular QTLs with GWAS, using the toy `protocol_example` dataset. Each method is a self-contained part with its own input files and output files.

## Pipeline position & prerequisites

This is the final, **advanced-analysis** notebook in the `protocol_example` toy bundle. It consumes artifacts produced by earlier notebooks, so run them first in this order:

1. `data_preprocessing/2_genotype_preprocessing` → produces `output/plink/` (QC'd genotypes).
2. `data_preprocessing/1_phenotype_preprocessing` → produces `output/phenotype/` (molecular phenotypes by chrom).
3. `data_preprocessing/4_covariates_preprocessing` → produces `output/covariate/` (covariates + PCs).
4. `qtl_association_finemapping/1_xqtl_association` → produces `output/phenotype/phenotype_by_chrom_for_cis/` region lists.
5. `qtl_association_finemapping/2_finemapping` → produces `output/finemapping/susie_twas/twas_weights/` (the precomputed TWAS prediction weights Part 1 needs).

Within this notebook the parts also chain to each other: earlier parts write to `output/twas/`, `output/susie_coloc/`, and `output/colocboost/`, which later parts read back.

**Pre-staged external inputs.** A few reference inputs are not produced by any notebook in this bundle, so they are provided ready-made under `input/`: `input/twas/` (TWAS meta/LD-block tables), `input/LD_sketch/` (RSS LD sketch reference), `input/susie_enloc_data/` (enloc priors & GWAS blocks), and `input/colocboost/` (ColocBoost GWAS meta). The specific input files each part needs are listed at the start of that part.


## Overview

1. **TWAS / MR** — transcriptome-wide association and Mendelian randomization.
2. **xQTL–GWAS enrichment** — global enrichment parameters used as colocalization priors.
3. **Pairwise colocalization (SuSiE-coloc)** — gene-by-gene xQTL–GWAS colocalization.
4. **ColocBoost** — multi-trait colocalization (xQTL-only and joint xQTL–GWAS modes).

The input files needed for each part are listed at the start of that part.

## 1. TWAS

**TWAS** tests gene–trait associations by combining precomputed molecular phenotype prediction weights with GWAS summary statistics, followed by Mendelian Randomization (MR) on candidate genes.

### Input Files

| File | Produced by | Used as |
|------|-------------|---------|
| `input/twas/protocol_example.twas.gwas_meta.tsv` | toy data prep | GWAS summary-statistics metadata |
| `input/twas/protocol_example.twas.xqtl_meta.tsv` | toy data prep | xQTL weight metadata (gene region + RDS weights) |
| `input/twas/protocol_example.twas.LD_blocks.chr22.bed` | toy data prep | analysis region definitions |
| `input/twas/protocol_example.twas.data_type_table.txt` | toy data prep | xQTL molecular phenotype type table |
| `input/LD_sketch/meta.tsv` | RSS LD sketch step | LD reference metadata |
| `output/finemapping/susie_twas/twas_weights/*.univariate_twas_weights.rds` | fine-mapping (2_finemapping) | TWAS prediction weights |

Key parameters: `--rsq_cutoff 0.01` (cross-validation R² threshold for predictable genes) and `--rsq_pval_cutoff` (cross-validation p-value threshold).

The TWAS step proceeds in four stages. First it **loads** the precomputed prediction weights from the RDS files. It then identifies **predictable genes** — those whose expression is reliably imputable based on cross-validation performance (controlled by `--rsq_cutoff` and `--rsq_pval_cutoff`). For those genes it computes **TWAS Z-scores** by combining the predicted expression with the GWAS summary statistics, and finally performs **Mendelian Randomization (MR)** on the candidate genes to assess potential causal effects of expression on the trait.

### **Step 1.** Load the TWAS prediction weights


In [16]:
# Load the TWAS prediction weights for the example gene
twas_weight = readRDS('output/finemapping/susie_twas/twas_weights/protocol_example_susie_twas.chr22_ENSG00000283047.univariate_twas_weights.rds')
str(twas_weight, max.level = 2)

List of 1
 $ ENSG00000283047:List of 1
  ..$ bulk_rnaseq_ENSG00000283047:List of 9


In [17]:
# R kernel
# Extract the fitted weights by method for the gene/context
twas_weight_by_method = twas_weight$ENSG00000283047$bulk_rnaseq_ENSG00000130538$twas_weights
str(twas_weight_by_method)

 NULL


### **Step 2.** Run TWAS and Mendelian Randomization


In [20]:
sos run pipeline/twas_ctwas.ipynb twas \
    --cwd output/twas \
    --name protocol_example \
    --gwas_meta_data input/twas/protocol_example.twas.gwas_meta.tsv \
    --ld_meta_data input/LD_sketch/meta.tsv \
    --ld_reference_sample_size 60 \
    --regions input/twas/protocol_example.twas.LD_blocks.chr22.bed \
    --xqtl_meta_data input/twas/protocol_example.twas.xqtl_meta.tsv \
    --xqtl_type_table input/twas/protocol_example.twas.data_type_table.txt \
    --rsq_pval_cutoff 1 \
    --rsq_cutoff 0.01

ParsingError: File contains parsing errors: 
	[line  2]: sos run pipeline/twas_ctwas.ipynb twas \
    --cwd output/twas \
    --name protocol_example \
    --gwas_meta_data input/twas/protocol_example.twas.gwas_meta.tsv \
    --ld_meta_data input/LD_sketch/meta.tsv \

Invalid statements: SyntaxError('invalid syntax', ('<string>', 1, 5, 'sos run pipeline/twas_ctwas.ipynb twas \\', 1, 8))

### Inspect the TWAS and MR results


In [ ]:
# Inspect the TWAS association results
zcat output/twas/twas/protocol_example.chr22_10000000_19000000.twas.tsv.gz | head

In [ ]:
# Inspect the Mendelian Randomization (MR) results
zcat output/twas/twas/protocol_example.chr22_10000000_19000000.mr_result.tsv.gz | head

### Output Files

| File | Contents |
|------|----------|
| `output/twas/twas/protocol_example.chr22_*.twas.tsv.gz` | Per-gene TWAS Z-scores and p-values |
| `output/twas/twas/protocol_example.chr22_*.mr_result.tsv.gz` | Mendelian Randomization results for candidate genes |

## 2. xQTL–GWAS enrichment

Estimates global enrichment parameters (a0, a1) between xQTL and GWAS signals and converts them into colocalization priors (p1, p2, p12) used by the colocalization steps that follow.

### Input Files

| File | Produced by | Used as |
|------|-------------|---------|
| `input/susie_enloc_data/protocol_example.enloc.gwas_meta.tsv` | SuSiE-enloc demo | GWAS fine-mapping metadata |
| `input/susie_enloc_data/protocol_example.enloc.xqtl_meta.tsv` | SuSiE-enloc demo | xQTL fine-mapping metadata |
| `input/susie_enloc_data/protocol_example.enloc.context_meta.tsv` | SuSiE-enloc demo | context / analysis-name map |
| `input/susie_enloc_data/*` | SuSiE-enloc demo | fine-mapping result objects |

### **Step 1.** Run xQTL–GWAS enrichment analysis


In [ ]:
sos run pipeline/SuSiE_enloc.ipynb xqtl_gwas_enrichment \
    --gwas-meta-data input/susie_enloc_data/protocol_example.enloc.gwas_meta.tsv \
    --xqtl-meta-data input/susie_enloc_data/protocol_example.enloc.xqtl_meta.tsv \
    --xqtl_finemapping_obj preset_variants_result susie_result_trimmed \
    --xqtl_varname_obj preset_variants_result variant_names \
    --gwas_finemapping_obj AD_Bellenguez_2022 RSS_QC_RAISS_imputed susie_result_trimmed \
    --gwas_varname_obj AD_Bellenguez_2022 RSS_QC_RAISS_imputed variant_names \
    --xqtl_region_obj region_info grange \
    --qtl-path input/susie_enloc_data \
    --gwas-path input/susie_enloc_data \
    --context-meta input/susie_enloc_data/protocol_example.enloc.context_meta.tsv \
    --cwd output/xqtl_gwas_enrichment


### Inspect the enrichment parameters


In [ ]:
# Inspect the enrichment parameters between xQTL and GWAS
# R kernel
coloc_enrichment = readRDS('output/xqtl_gwas_enrichment/protocol_example.enloc.Knight_eQTL_brain.enrichment.rds')
str(coloc_enrichment)

### Output Files

| File | Contents |
|------|----------|
| `output/xqtl_gwas_enrichment/*.enrichment.rds` | Enrichment parameters (a0, a1) and derived colocalization priors (p1, p2, p12) |

## 3. Pairwise colocalization (SuSiE-coloc)

Runs gene-by-gene xQTL–GWAS colocalization with `susie_coloc`, using either the enrichment-derived priors from Part 2 or default priors (`--skip-enrich`). LD metadata enables variant-level credible sets.

### Input Files

| File | Produced by | Used as |
|------|-------------|---------|
| `input/susie_enloc_data/protocol_example.enloc.gwas_meta.tsv` | SuSiE-enloc demo | GWAS fine-mapping metadata |
| `input/susie_enloc_data/protocol_example.enloc.xqtl_meta.tsv` | SuSiE-enloc demo | xQTL fine-mapping metadata (with overlap info) |
| `input/susie_enloc_data/protocol_example.enloc.context_meta.tsv` | SuSiE-enloc demo | context / analysis-name map |
| `input/ld_reference/protocol_example.ld_meta_file.tsv` | toy data prep | LD reference metadata for credible sets |
| `input/susie_enloc_data/*` | SuSiE-enloc demo | fine-mapping result objects |

### **Step 1.** Run pairwise colocalization (SuSiE-coloc)


In [ ]:
sos run pipeline/SuSiE_enloc.ipynb susie_coloc \
    --gwas-meta-data input/susie_enloc_data/protocol_example.enloc.gwas_meta.tsv \
    --xqtl-meta-data input/susie_enloc_data/protocol_example.enloc.xqtl_meta.tsv \
    --xqtl_finemapping_obj preset_variants_result susie_result_trimmed \
    --xqtl_varname_obj preset_variants_result variant_names \
    --gwas_finemapping_obj AD_Bellenguez_2022 RSS_QC_RAISS_imputed susie_result_trimmed \
    --gwas_varname_obj AD_Bellenguez_2022 RSS_QC_RAISS_imputed variant_names \
    --xqtl_region_obj region_info grange \
    --qtl-path input/susie_enloc_data \
    --skip-enrich \
    --gwas-path input/susie_enloc_data \
    --context-meta input/susie_enloc_data/protocol_example.enloc.context_meta.tsv \
    --ld_meta_file_path input/ld_reference/protocol_example.ld_meta_file.tsv \
    --cwd output/susie_coloc


### Inspect the colocalization results


In [ ]:
# Inspect the colocalization results for one gene
# R kernel
coloc_result = readRDS('output/susie_coloc/susie_coloc/protocol_example.enloc.Knight_eQTL_brain@ENSG00000142798.coloc.rds')
str(coloc_result)

### Output Files

| File | Contents |
|------|----------|
| `output/susie_coloc/susie_coloc/*@<gene>.coloc.rds` | Per gene-context colocalization stats and posterior probabilities |
| `output/susie_coloc/*.coloc_res` | Variant-level credible-set results (when LD metadata is supplied) |

Each `.coloc.rds` is a nested R list with `summary` (posterior probabilities for the five coloc hypotheses), `results` (SNP-level PP.H4), `priors`, and `analysis_region`. **PP.H4** is the probability that both traits share a single causal variant — higher values mean stronger colocalization evidence.

Each `.coloc.rds` reports posterior probabilities for five mutually exclusive colocalization hypotheses for a given gene-trait pair: **PP.H0** (neither trait has an association in the region), **PP.H1** (only the xQTL is associated), **PP.H2** (only the GWAS trait is associated), **PP.H3** (both are associated but driven by *different* causal variants), and **PP.H4** (both are associated and share a *single* causal variant). A high **PP.H4** is the evidence for colocalization — it indicates the molecular and complex traits are likely driven by the same underlying variant.

## 4. ColocBoost

Multi-trait colocalization that jointly analyzes molecular traits, optionally together with GWAS. Two modes are shown: **xQTL-only** (`--no-separate-gwas`) and **joint xQTL–GWAS** (`--separate-gwas`).

### Input Files

| File | Produced by | Used as |
|------|-------------|---------|
| `output/plink/protocol_example.genotype.merged.plink_qc.bed` | genotype preprocessing | merged genotypes |
| `output/phenotype/.../bulk_rnaseq.phenotype_by_chrom_files.region_list.txt` | phenotype preprocessing | phenotype region list |
| `output/covariate/protocol_example...Marchenko_PC.gz` | covariate preprocessing | covariates |
| `input/reference_data/TAD/TADB_enhanced_cis.bed` | reference data | cis association windows |
| `input/LD_sketch/meta.tsv` | RSS LD sketch step | LD reference metadata (joint mode) |
| `input/colocboost/gwas_meta.txt` | toy data prep | GWAS metadata, tab-separated (joint mode) |

### **Step 1.** xQTL-only ColocBoost (`--no-separate-gwas`)


In [ ]:
sos run pipeline/colocboost.ipynb colocboost \
    --name test_coloc_boost_xqtl_only \
    --cwd output/colocboost \
    --genoFile output/plink/protocol_example.genotype.merged.plink_qc.bed \
    --phenoFile output/phenotype/phenotype_by_chrom_for_cis/bulk_rnaseq.phenotype_by_chrom_files.region_list.txt \
    --covFile output/covariate/protocol_example.rnaseq.bed.protocol_example.covariates.protocol_example.genotype.merged.plink_qc.plink_qc.prune.pca.Marchenko_PC.gz \
    --customized-association-windows input/reference_data/TAD/TADB_enhanced_cis.bed \
    --no-separate-gwas --xqtl-coloc \
    --region-name ENSG00000130538 \
    --phenotype-names trait_A


### **Step 2.** Joint xQTL–GWAS ColocBoost (`--separate-gwas`)


In [ ]:
sos run pipeline/colocboost.ipynb colocboost \
    --name colocboost_gwas \
    --cwd output/colocboost_gwas \
    --genoFile output/plink/protocol_example.genotype.merged.plink_qc.bed \
    --phenoFile output/phenotype/phenotype_by_chrom_for_cis/bulk_rnaseq.phenotype_by_chrom_files.region_list.txt \
    --covFile output/covariate/protocol_example.rnaseq.bed.protocol_example.covariates.protocol_example.genotype.merged.plink_qc.plink_qc.prune.pca.Marchenko_PC.gz \
    --customized-association-windows input/reference_data/TAD/TADB_enhanced_cis.bed \
    --ld-meta-data input/LD_sketch/meta.tsv \
    --gwas-meta-data input/colocboost/gwas_meta.txt \
    --separate-gwas --xqtl-coloc \
    --region-name ENSG00000130538 \
    --phenotype-names trait_A


### Inspect the ColocBoost output


In [ ]:
# Inspect the ColocBoost output
# R kernel
colocboost = readRDS('output/colocboost_gwas/colocboost/colocboost_gwas.chr22_ENSG00000130538.cb_xqtl_synthetic_gwas_chr22.rds')
str(colocboost)

### Output Files

| File | Contents |
|------|----------|
| `output/colocboost/.../cb_xqtl_*.rds` | xQTL-only ColocBoost result per gene |
| `output/colocboost_gwas/.../cb_xqtl_synthetic_gwas_*.rds` | Joint xQTL–GWAS ColocBoost result per gene |

Each `.rds` is a named list (class `colocboost`) keyed by gene. The most useful elements are `cos_summary` (colocalized signals, empty if none), `cos_details` (details per colocalized signal), `ucos_details` (uncolocalized signals — many of these suggest GWAS signals independent of molecular QTLs), and `data_info` / `model_info` (traits analyzed and model convergence).